In [2]:
# ============================================================
# Sensitivity Test — run all 8 test cases programmatically
# ============================================================

import pandas as pd
import numpy as np
import joblib
import json

pipeline = joblib.load('../models/preprocessing_pipeline.joblib')
kmeans_final = joblib.load('../models/kmeans_model.joblib')
classifier = joblib.load('../models/classifier.joblib')
with open('../models/model_config.json') as f:
    config = json.load(f)

def predict(patient_dict):
    patient_df = pd.DataFrame([patient_dict])
    X_proc = pipeline.transform(patient_df)
    cluster = kmeans_final.predict(X_proc)
    X_final = np.hstack([X_proc, cluster.reshape(-1, 1)])
    prob = classifier.predict_proba(X_final)[:, 1][0]

    if prob < config['risk_tier_low_cutoff']:
        tier = "Low"
    elif prob < config['risk_tier_high_cutoff']:
        tier = "Medium"
    else:
        tier = "High"
    return round(float(prob), 4), tier, int(cluster[0])

# Base template — every test case starts here and overrides specific fields
def base_patient(**overrides):
    p = {
        'race': 'Caucasian', 'gender': 'Female', 'age_ordinal': 4,
        'admission_type_id': 1, 'discharge_disposition_id': 1, 'admission_source_id': 1,
        'time_in_hospital': 3, 'medical_specialty': 'Unknown',
        'num_lab_procedures': 40, 'num_procedures': 1, 'num_medications': 15,
        'number_outpatient': 0, 'number_emergency': 0, 'number_inpatient': 0,
        'diag_1_group': 'Other', 'diag_2_group': 'Other', 'diag_3_group': 'Other',
        'number_diagnoses': 7, 'max_glu_serum': 'None', 'A1Cresult': 'None',
        'change': 'No', 'diabetesMed': 'Yes',
        'total_prior_utilization': 0, 'num_meds_changed': 0,
    }
    p.update(overrides)
    p['total_prior_utilization'] = p['number_outpatient'] + p['number_emergency'] + p['number_inpatient']
    p['num_meds_changed'] = 1 if p['change'] == 'Ch' else 0
    return p

In [3]:
tests = {
    "1. Baseline (defaults)": base_patient(),

    "2. Young + minimal": base_patient(
        age_ordinal=2, time_in_hospital=1, num_lab_procedures=5, num_procedures=0,
        num_medications=1, number_diagnoses=1, diag_1_group='Respiratory'
    ),

    "3. Elderly + minimal (age isolated)": base_patient(
        age_ordinal=8, time_in_hospital=1, num_lab_procedures=5, num_procedures=0,
        num_medications=1, number_diagnoses=1, diag_1_group='Respiratory'
    ),

    "4-corrected. Realistic high-risk profile": base_patient(
    age_ordinal=7, time_in_hospital=6, num_medications=15,
    number_inpatient=4,
    discharge_disposition_id=22,   # ← rehab transfer, NOT home
    diag_1_group='Musculoskeletal', number_diagnoses=8
    ),
    "4b. Same but discharge=3 (SNF)": base_patient(
        age_ordinal=7, time_in_hospital=6, num_medications=15,
        number_inpatient=4,
        discharge_disposition_id=3,
        diag_1_group='Musculoskeletal', number_diagnoses=8
    ),
        "5. High medications only": base_patient(
            age_ordinal=4, num_medications=60, time_in_hospital=2,
        number_inpatient=0, diag_1_group='Other'
    ),

    "6. High lab procedures only": base_patient(
        age_ordinal=4, num_lab_procedures=120, time_in_hospital=2,
        number_inpatient=0, diag_1_group='Other'
    ),

    "7. Out-of-distribution inpatient (15)": base_patient(
        age_ordinal=7, time_in_hospital=6, num_medications=15, number_inpatient=15,
        diag_1_group='Musculoskeletal', number_diagnoses=8
    ),

    "8a. Diagnosis - Respiratory": base_patient(
        age_ordinal=2, time_in_hospital=1, num_medications=1,
        number_diagnoses=1, diag_1_group='Respiratory'
    ),
    "8b. Diagnosis - Circulatory": base_patient(
        age_ordinal=2, time_in_hospital=1, num_medications=1,
        number_diagnoses=1, diag_1_group='Circulatory'
    ),
    "8c. Diagnosis - Musculoskeletal": base_patient(
        age_ordinal=2, time_in_hospital=1, num_medications=1,
        number_diagnoses=1, diag_1_group='Musculoskeletal'
    ),
}

results = []
for name, patient in tests.items():
    prob, tier, cluster = predict(patient)
    results.append({'Test': name, 'Probability': f"{prob:.1%}", 'Tier': tier, 'Cluster': cluster})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

                                    Test Probability   Tier  Cluster
                  1. Baseline (defaults)        4.9%    Low        3
                      2. Young + minimal        2.2%    Low        3
     3. Elderly + minimal (age isolated)        2.6%    Low        2
4-corrected. Realistic high-risk profile       26.1%   High        2
          4b. Same but discharge=3 (SNF)       17.6%   High        2
                5. High medications only        3.3%    Low        0
             6. High lab procedures only        6.6%    Low        3
   7. Out-of-distribution inpatient (15)       10.0% Medium        2
             8a. Diagnosis - Respiratory        2.2%    Low        3
             8b. Diagnosis - Circulatory        3.6%    Low        3
         8c. Diagnosis - Musculoskeletal        2.5%    Low        3


In [4]:
# Save results for the report
results_df.to_csv('../reports/predictor_sensitivity_test.csv', index=False)
print("\nSaved: reports/predictor_sensitivity_test.csv")


Saved: reports/predictor_sensitivity_test.csv


In [5]:
df = pd.read_csv('../data/processed/cleaned_data.csv')

print("=== Readmission rate by discharge_disposition_id ===")
print(df.groupby('discharge_disposition_id')['readmitted_binary'].agg(['mean', 'count']).sort_values('mean', ascending=False))

=== Readmission rate by discharge_disposition_id ===
                              mean  count
discharge_disposition_id                 
12                        0.500000      2
15                        0.450000     40
28                        0.355556     90
22                        0.263121   1410
9                         0.222222      9
5                         0.205915    913
2                         0.137752   1539
3                         0.133880   8784
4                         0.103512    541
18                        0.101455   2474
8                         0.095890     73
7                         0.095355    409
6                         0.095186   8289
1                         0.069454  44317
25                        0.061697    778
24                        0.040000     25
23                        0.030769    260
16                        0.000000      3
10                        0.000000      6
17                        0.000000      8
27                     

In [6]:
print("=== Readmission rate by number_inpatient ===")
print(df.groupby('number_inpatient')['readmitted_binary'].agg(['mean', 'count']).sort_values('mean', ascending=False).head(15))

=== Readmission rate by number_inpatient ===
                      mean  count
number_inpatient                 
12                1.000000      2
10                0.800000      5
11                0.500000      2
7                 0.473684     19
9                 0.428571      7
6                 0.400000     55
5                 0.284314    102
3                 0.237581    463
4                 0.236842    228
8                 0.230769     13
2                 0.185210   1501
1                 0.128926   5794
0                 0.081173  61782


In [8]:
import pandas as pd
import numpy as np
import joblib

pipeline = joblib.load('../models/preprocessing_pipeline.joblib')
kmeans_final = joblib.load('../models/kmeans_model.joblib')

df = pd.read_csv('../data/processed/cleaned_data.csv')
sample = df.sample(5000, random_state=42).copy()

# --- Re-apply Phase 3 feature engineering (missing from raw cleaned_data.csv) ---
sample['total_prior_utilization'] = (
    sample['number_outpatient'] + sample['number_emergency'] + sample['number_inpatient']
)

med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
            'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
            'miglitol', 'troglitazone', 'tolazamide', 'examide',
            'citoglipton', 'insulin', 'glyburide-metformin',
            'glipizide-metformin', 'glimepiride-pioglitazone',
            'metformin-rosiglitazone', 'metformin-pioglitazone']
med_cols = [c for c in med_cols if c in sample.columns]

sample['num_meds_changed'] = sample[med_cols].apply(
    lambda row: sum(1 for v in row if v in ['Up', 'Down']), axis=1
)

# --- Now transform ---
drop_cols = [c for c in ['patient_nbr', 'readmitted', 'readmitted_binary', 'cluster'] if c in sample.columns]
X = sample.drop(columns=drop_cols)

X_proc = pipeline.transform(X)
clusters = kmeans_final.predict(X_proc)
sample['cluster'] = clusters

print("Cluster sizes:")
print(sample['cluster'].value_counts().sort_index())
print()

profile_cols = ['time_in_hospital', 'num_lab_procedures', 'num_medications',
                 'number_diagnoses', 'number_inpatient']
profile_cols = [c for c in profile_cols if c in sample.columns]
print(sample.groupby('cluster')[profile_cols].mean().round(2))
print()

print("Readmission rate by cluster:")
print(sample.groupby('cluster')['readmitted_binary'].mean().round(3))

Cluster sizes:
cluster
0     739
1     835
2    2105
3    1321
Name: count, dtype: int64

         time_in_hospital  num_lab_procedures  num_medications  \
cluster                                                          
0                    8.07               57.65            27.23   
1                    4.56               44.30            17.10   
2                    3.46               39.46            12.99   
3                    3.01               39.45            12.49   

         number_diagnoses  number_inpatient  
cluster                                      
0                    8.22              0.21  
1                    7.75              0.18  
2                    7.37              0.19  
3                    6.27              0.15  

Readmission rate by cluster:
cluster
0    0.104
1    0.113
2    0.099
3    0.077
Name: readmitted_binary, dtype: float64


In [9]:
# Generate a valid test file
test_sample = df.drop(columns=['patient_nbr', 'readmitted', 'readmitted_binary']).sample(20, random_state=1)
test_sample.to_csv('../data/processed/test_batch_valid.csv', index=False)
print("Saved: data/processed/test_batch_valid.csv — upload this in the Batch Upload page")

# Generate a broken test file (missing a required column) to confirm graceful failure
broken_sample = test_sample.drop(columns=['age_ordinal'])
broken_sample.to_csv('../data/processed/test_batch_broken.csv', index=False)
print("Saved: data/processed/test_batch_broken.csv — upload this too, should show an error, not crash")

Saved: data/processed/test_batch_valid.csv — upload this in the Batch Upload page
Saved: data/processed/test_batch_broken.csv — upload this too, should show an error, not crash


In [10]:
import os
print(os.path.exists('../data/processed/test_batch_valid.csv'))
print(os.path.exists('../data/processed/test_batch_broken.csv'))

True
True
